# 0. [write] Remove foreign companies

In [ ]:
import pandas as pd
import ibis
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs()
con = ibis.duckdb.connect(str(dirs.db_path))
table_names = ["fame_fixed", "fame_yearly"]
overwrite_flag = True

for t in table_names:
    table_raw = con.table(t)
    # Check if the table exists in the database
    if table_raw is None:
        raise ValueError(f"Table {t} does not exist in the database. Please check the table name and try again.")
    fixed_count_start = table_raw.count().execute()

    # ==========================================
    # 1. EXTRACT & COUNT COUNTRY CODES
    # ==========================================
    print("🔍 Extracting country codes and generating counts...")

    # Use regex to grab any leading letters or the '#' symbol
    table_with_cc = table_raw.mutate(
        country_code = table_raw.registered_number.re_extract(r'^([A-Za-z\#]+)', 1)
    )

    # Aggregate and execute to Pandas
    cc_counts = (
        table_with_cc.group_by("country_code")
        .aggregate(count=table_with_cc.count())
        .execute()
    )

    # ==========================================
    # 2. FILTER OUT FOREIGN FIRMS
    # ==========================================
    # Define standard UK prefixes (England/Wales are pure numbers, handled by isnull)
    uk_prefixes = [
        '',   # England & Wales (pure numbers)
        'NI', # Northern Ireland Company (post-partition)
        'SC', # Scottish Company
        'OC', # Limited Liability Partnership - LLP (England & Wales)
        'SO', # Limited Liability Partnership - LLP (Scotland)
        'NC', # Limited Liability Partnership - LLP (Northern Ireland)
        'LP', # Limited Partnership (England & Wales)
        'SL', # Limited Partnership (Scotland)
        'ZC', # Unregistered Companies (Section 1043) for England & Wales
        'SZ', # Scottish Unregistered Companies (Section 1043)
        'SG', # Scottish Qualifying Partnership
        'CE', # Charitable Incorporated Organisation (England & Wales)
        'CS', # Scottish Charitable Incorporated Organisation
        'R',  # Older Northern Ireland company (no longer issued)
        'IP'  # Industrial and Provident Societies (cooperatives)
    ]
    non_uk_prefixes = [
        'IE', # Ireland
        'JE', # Jersey
        'IM', # Isle of Man
        'GG', # Guernsey
        'GI', # Gibraltar,
        'SE'  # Société Européenne (European Company),
        '#',  # Foreign legal entites that are traded on LSEG
    ]

    # Format the NaN values (pure numbers) for clean viewing
    cc_counts["country_code"] = cc_counts["country_code"].fillna("None at all (All numbers)")
    cc_counts = cc_counts.sort_values(by="count", ascending=False).reset_index(drop=True)
    uk_cc_counts = cc_counts[cc_counts["country_code"].isin(uk_prefixes + ["None at all (All numbers)"])]
    non_uk_cc_counts = cc_counts[cc_counts["country_code"].isin(non_uk_prefixes)]
    non_uk_count = non_uk_cc_counts["count"].sum()
    non_uk_total = pd.DataFrame({"country_code": ["Total Non-UK"], "count": non_uk_count})
    non_uk_cc_counts = pd.concat([non_uk_cc_counts, non_uk_total], ignore_index=True)
    print(f"📊 UK prefix Counts in {t}:")
    display(uk_cc_counts)
    print(f"📊 Non-UK prefix Counts in {t}:")
    display(non_uk_cc_counts)

    # Keep rows where prefix is missing (pure numbers) OR is a known UK prefix
    # Then immediately drop the temporary 'country_code' column so it doesn't pollute the schema
    table_filtered = table_with_cc.filter(
        # Remove rows where country_code is not in non_uk_prefixes using ibis method
        table_with_cc.country_code.isin(uk_prefixes)
    )
    table_skinny = table_filtered.drop("country_code")

    # Print and compare row count
    filtered_count = table_skinny.count().execute()
    print(f"Number of rows in fame_fixed after filtering: {filtered_count:,}")

    if fixed_count_start - filtered_count != non_uk_count:
        print(f"Pre-filtering count: {fixed_count_start:,}, Post-filtering count: {filtered_count:,}, Non-UK count: {non_uk_count:,}")
        # raise ValueError("Row count mismatch after filtering. Please check the filtering logic.")
    
    if overwrite_flag:
        print(f"💾 Safely overwriting {t}...")
        con.create_table(f"{t}_clean", table_skinny, overwrite=True)
        con.drop_table(t)
        con.create_table(t, con.table(f"{t}_clean"), overwrite=True)
        con.drop_table(f"{t}_clean")
        print(f"✅ {t} updated successfully.")

🔍 Extracting country codes and generating counts...
📊 UK prefix Counts in fame_fixed:


,country_code,count
0,,7686407
1,SC,429206
2,NI,99108
3,SO,254
4,R,152
7,ZC,70
11,SZ,4
13,IP,2


📊 Non-UK prefix Counts in fame_fixed:


,country_code,count
0,JE,94
1,IM,50
2,GG,40
3,GI,4
4,Total Non-UK,188


Number of rows in fame_fixed after filtering: 8,215,203
Pre-filtering count: 8,215,526, Post-filtering count: 8,215,203, Non-UK count: 188
💾 Safely overwriting fame_fixed...
✅ fame_fixed updated successfully.
🔍 Extracting country codes and generating counts...
📊 UK prefix Counts in fame_fixed:


,country_code,count
0,,41765701
2,SC,2431837
3,NI,662410
4,SO,2943
5,R,1762
7,ZC,408
12,SZ,54
14,IP,12


📊 Non-UK prefix Counts in fame_fixed:


,country_code,count
0,IE,2589759
1,JE,421
2,IM,272
3,GG,205
4,GI,29
5,Total Non-UK,2590686


Number of rows in fame_fixed after filtering: 44,865,127
Pre-filtering count: 47,456,501, Post-filtering count: 44,865,127, Non-UK count: 2,590,686
💾 Safely overwriting fame_yearly...
✅ fame_yearly updated successfully.


# 1. Filter on fame_fixed

In [ ]:
# Load fame_fixed table from duckdb
# Print the number of rows
import ibis
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs()
con = ibis.duckdb.connect(str(dirs.db_path))
fame_fixed = con.table("fame_fixed")
if fame_fixed is None:
    raise ValueError("fame_fixed table not found in the database.")

fixed_count_start = fame_fixed.count().execute()
print(f"Number of rows in fame_fixed before filtering: {fixed_count_start:,}")

fame_fixed_running = fame_fixed
con.create_table("fame_fixed_filtered", fame_fixed_running, overwrite=True)
print(f"✅ fame_fixed_filtered created with {fame_fixed_running.count().execute():,} rows.")

Number of rows in fame_fixed before filtering: 8,215,203


# 2. Filter on fame_yearly

In [ ]:
# Load fame_yearly table from duckdb
# Print the number of rows
import ibis
from utils.f_0_dirs import get_data_dirs

table_name = "fame_yearly"
dirs = get_data_dirs()
con = ibis.duckdb.connect(str(dirs.db_path))
fame_yearly = con.table(table_name)
if fame_yearly is None:
    raise ValueError(f"{table_name} table not found in the database.")

yearly_count_start = fame_yearly.count().execute()
print(f"Number of rows in {table_name} before filtering: {yearly_count_start:,}")

fame_yearly_running = fame_yearly

Number of rows in fame_yearly_filtered before filtering: 1,171,378


### Drop small companies (<10 employees)

In [12]:
# Drop any row in fame_yearly where employees is NaN or <10, as these are likely to be small companies that are not relevant for our analysis.

fame_yearly_no_small = fame_yearly_running.filter(
    (fame_yearly_running['employees'].isnull() == False) & (fame_yearly_running['employees'] >= 10)
)
yearly_count_no_small = fame_yearly_no_small.count().execute()
print(f"Number of rows in fame_yearly after filtering small companies: {yearly_count_no_small:,}")

fame_yearly_running = fame_yearly_no_small

Number of rows in fame_yearly after filtering small companies: 1,171,378


### Drop missing data (for productivity)
- GVA (Lars): renumeration_employees + ebitda
- GVA (bottom-up): profit_loss_pretax + interest_paid + depreciation + remuneration_employees
- Productivity: GVA / employees
- Average wage: renumeration_employees / employees

In [13]:
# - GVA (Lars): renumeration_employees + ebitda
# - GVA (bottom-up): profit_loss_pretax + interest_paid + depreciation + remuneration_employees
# - Productivity: GVA / employees
# - Average wage: renumeration_employees / employees
# Drop any row which is missing (renumeration_employees and ebitda) or (profit_loss_pretax, interest_paid, depreciation, remuneration_employees), as these are needed to calculate gva.

fame_yearly_no_missing = fame_yearly_running.filter(
    (
        (fame_yearly_running['remuneration_employees'].isnull() == False) &
        (fame_yearly_running['ebitda'].isnull() == False)
    ) | (
        (fame_yearly_running['profit_loss_pretax'].isnull() == False) &
        (fame_yearly_running['interest_paid'].isnull() == False) &
        (fame_yearly_running['depreciation'].isnull() == False) &
        (fame_yearly_running['remuneration_employees'].isnull() == False)
    )
)
yearly_count_no_missing = fame_yearly_no_missing.count().execute()
print(f"Number of rows in fame_yearly after filtering missing data: {yearly_count_no_missing:,}")

fame_yearly_running = fame_yearly_no_missing

Number of rows in fame_yearly after filtering missing data: 1,171,378


### Output filtered yearly table

In [14]:
# Write fame_yearly_running to a new table in duckdb called fame_yearly_filtered
con.create_table("fame_yearly_filtered", fame_yearly_running, overwrite=True)
print(f"✅ fame_yearly_filtered created with {fame_yearly_running.count().execute():,} rows.")

✅ fame_yearly_filtered created with 1,171,378 rows.


# 3. Write filter to both databases

In [ ]:
import ibis
from utils.f_0_dirs import get_data_dirs

tables = ["fame_fixed_filtered", "fame_yearly_filtered"]

dirs = get_data_dirs(segment="build")
con = ibis.duckdb.connect(str(dirs.db_path))
con.raw_sql("PRAGMA memory_limit='12GB'")

fame_fixed_running = con.table(tables[0])
fame_yearly_running = con.table(tables[1])

fixed_cols = list(fame_fixed_running.columns)
yearly_cols = list(fame_yearly_running.columns)

start_fixed_count = fame_fixed_running.count().execute()
start_yearly_count = fame_yearly_running.count().execute()
print(f"📊 Initial firm count (fame_fixed): {start_fixed_count:,}")
print(f"📊 Initial panel row count (fame_yearly): {start_yearly_count:,}")

fame_fixed_filtered = fame_fixed_running.semi_join(fame_yearly_running, "registered_number")
fame_yearly_filtered = fame_yearly_running.semi_join(fame_fixed_running, "registered_number")

end_fixed_count = fame_fixed_filtered.count().execute()
end_yearly_count = fame_yearly_filtered.count().execute()
print(f"📊 Final firm count (fame_fixed): {end_fixed_count:,}")
print(f"📊 Final panel row count (fame_yearly): {end_yearly_count:,}")

📊 Initial firm count (fame_fixed): 152,379
📊 Initial panel row count (fame_yearly): 1,128,490
📊 Final firm count (fame_fixed): 152,379
📊 Final panel row count (fame_yearly): 1,128,490


In [ ]:
# ==========================================
# SAFE TABLE OVERWRITE PROCEDURE
# ==========================================

if start_fixed_count != end_fixed_count:
    print("💾 Safely overwriting fame_fixed...")
    con.create_table(tables[0] + "_clean", fame_fixed_filtered, overwrite=True)
    con.drop_table(tables[0])
    con.create_table(tables[0], con.table(tables[0] + "_clean"), overwrite=True)
    con.drop_table(tables[0] + "_clean")
    print(f"✅ {tables[0]} updated successfully.")

if start_yearly_count != end_yearly_count:
    print("💾 Safely overwriting fame_yearly...")
    con.create_table(tables[1] + "_clean", fame_yearly_filtered, overwrite=True)
    con.drop_table(tables[1])
    con.create_table(tables[1], con.table(tables[1] + "_clean"), overwrite=True)
    con.drop_table(tables[1] + "_clean")
    print(f"✅ {tables[1]} updated successfully.")

# Verification
final_fixed_count = con.table(tables[0]).count().execute()
final_yearly_count = con.table(tables[1]).count().execute()
print(f"📊 Final firm count ({tables[0]}): {final_fixed_count:,}")
print(f"📊 Final panel row count ({tables[1]}): {final_yearly_count:,}")

💾 Safely overwriting fame_fixed...
✅ fame_fixed updated successfully.
💾 Safely overwriting fame_yearly...
✅ fame_yearly_filtered updated successfully.
📊 Final firm count (fame_fixed): 152,379
📊 Final panel row count (fame_yearly_filtered): 1,265,680
